In [1]:
%load_ext autoreload
%autoreload 2

In [14]:
import mappy as mp
import pyranges as pr
import pandas as pd
from datasets import load_dataset
from Bio import SeqIO
from pathlib import Path
from intervaltree import IntervalTree
from collections import Counter
from src.utils.canonicalization import parse_transcribed_features_from_gff

# Paths
fasta_path = "../../data/reference_sr.mmi"
gff_path = "../../data/hg38/gencode.v50.annotation.gff3"

print("Loading data...")
aligner = mp.Aligner(fasta_path)

Loading data...


In [3]:
gr = pr.read_gff3(gff_path)
# Convert level to int to avoid comparison issues
gr.level = gr.level.astype(int)

In [4]:
for c in gr.chromosomes:
    print(c)

chr1
chr2
chr3
chr4
chr5
chr6
chr7
chr8
chr9
chr10
chr11
chr12
chr13
chr14
chr15
chr16
chr17
chr18
chr19
chr20
chr21
chr22
chrM
chrX
chrY


In [5]:
dataset = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks")
task_name = "splice_sites_all" # "enhancers", "splice_sites_all", etc.
filtered = dataset.filter(lambda x: x["task"] == task_name)
print(f"Dataset filtered for {task_name}. Size: {len(filtered['train'])}")

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Dataset filtered for splice_sites_all. Size: 27000


In [6]:
canonicalization_segments = {}  # chrom -> { 'positive': IntervalTree, 'negative': IntervalTree }

print("Processing GFF for canonicalization segments...")
gff_df = gr.df
for chrom, group in gff_df.groupby("Chromosome"):
    pos, neg = parse_transcribed_features_from_gff(
        group, chrom, annotations_level=2, include_cdjv_segments=True, include_pseudogenes=True
    )
    canonicalization_segments[chrom] = {
        "positive": IntervalTree.from_tuples(pos),
        "negative": IntervalTree.from_tuples(neg)
    }
print("Done.")

Processing GFF for canonicalization segments...
Done.


In [7]:
print(len(canonicalization_segments["chr1"]["positive"]))
print(len(canonicalization_segments["chr1"]["negative"]))

2677
2567


In [19]:
print(f"Running detailed loop for first 100 sequences...")
stats = Counter()
for i, seq in enumerate(filtered['train']['sequence']):
    stats['total'] += 1
    hit = next((x for x in aligner.map(seq) if x.is_primary), None)
    if hit is None:
        stats['no_hit'] += 1
        continue
    if hit.mapq < 10:
        print(hit.mapq)
        raise ValueError("Bad hit")
    
    stats['has_hit'] += 1
    hit_id = hit.ctg
    if hit_id not in canonicalization_segments:
        stats['chrom_mismatch'] += 1
        print(hit.NM)
        continue

    segs = canonicalization_segments[hit_id]
    strand_key = 'positive' if hit.strand == 1 else 'negative'
    tree = segs[strand_key]

    if not tree.overlap(hit.r_st, hit.r_en):
        stats['miss'] += 1
        # Detailed diagnostic from the join (which was done without strand constraint)
        local_feats = df_diag[df_diag["hit_index"] == i]
        if not local_feats.empty:
            types = local_feats["Feature"].value_counts().to_dict()
            genes = local_feats["gene_name"].unique().tolist()
            # Strand_b is the strand of the feature in GFF (+ or -)
            hit_strand_str = "+" if hit.strand == 1 else "-"
            same_strand_count = (local_feats["Strand_b"] == hit_strand_str).sum()
            
            print(f"[MISS] {hit_id}:{hit.r_st}-{hit.r_en} strand={hit_strand_str} | "
                  f"types: {types} | genes: {genes} | same_strand: {same_strand_count}/{len(local_feats)}")
        else:
            print(f"[MISS] {hit_id}:{hit.r_st}-{hit.r_en} strand={'+' if hit.strand==1 else '-'} → intergenic")
    else:
        stats['overlap'] += 1

print(f"\nSummary: {dict(stats)}")

Running detailed loop for first 100 sequences...
7


ValueError: Bad hit